# B v4 — Clustering dari fitur PERUBAHAN (Fahmi)

**Masalah v3:** cluster dibentuk dari embedding `color_pre + color_post` (scene). Akibatnya cluster mengelompokkan **jenis lokasi**
(kota padat, hutan tropis) dan nama ditebak dari lokasi. Skor publik tertahan di ±0,39 walau classifier meniru nama cluster dengan F1 0,87.

**Perbaikan v4:** cluster dari `dino_changed_embed` (Hafizh v3.1): token DINOv2 post yang dibobot peta perubahan pre→post,
jadi yang dikelompokkan adalah **seperti apa area yang berubah**, bukan seperti apa lokasinya.

Pemilihan fitur (cek tanpa label, K=40, PCA 64, kNN 15, seed 42):

| Opsi | η² perubahan ↑ | η² scene ↓ | bias artefak maks | ARI noise | Keputusan |
|---|---|---|---|---|---|
| pre+post (v3) | 0,121 | 0,224 | 0,149 | 0,985 | lama |
| changed + diff | 0,162 | 0,178 | 0,139 | 0,821 | gagal gate noise |
| **changed** | **0,163** | 0,171 | 0,143 | **0,989** | **dipakai** |

Folder lama (`v3*`) tidak diubah. Output ke `v4_perubahan/`. Nama kelas diisi tim dari montase di `lembar_bersama.csv` (FAQ no. 7).

## 0. Library & pengaturan

In [1]:
import json, os, hashlib, platform
from datetime import datetime
import numpy as np
import pandas as pd
import joblib, sklearn
from sklearn.decomposition import PCA
from sklearn.cluster import SpectralClustering
from sklearn.metrics import adjusted_rand_score
from PIL import Image, ImageDraw, ImageFont

# --- Path (relatif terhadap folder fahmi/) ---
FITUR_DIR  = "../hafizh/v3.1"
ID_ACUAN   = "v3_revisi2/events.parquet"   # hanya untuk urutan Id (= sample_submission) dan perbandingan
DATA_DIR   = "../Dataset"
OUTPUT_DIR = "v4_perubahan"

# --- Fitur & clustering ---
FITUR = "changed"       # pilihan: "changed", "diff", "changed+diff", "prepost"
SEED, PCA_DIM, ROUND, K, KNN = 42, 64, 4, 40, 15

# --- Gate ---
GATE_ARI_NOISE, GATE_BIAS_MAX, GATE_MIN_SIZE, NOISE_SKALA = 0.95, 0.15, 20, 1e-3

# --- Montase (sama dengan v3) ---
N_DEKAT, N_ACAK, TILE, PASANG_PER_BARIS, JPEG_Q = 8, 8, 320, 2, 85
KELAS = ["Earthquake", "Explosion", "Fire", "Hurricane", "Tsunami", "Volcano"]

assert not os.path.exists(f"{OUTPUT_DIR}/meta.json"), "v4 sudah ada -> buat folder versi baru, jangan timpa"
os.makedirs(f"{OUTPUT_DIR}/montase", exist_ok=True)
print("sklearn", sklearn.__version__, "| output:", OUTPUT_DIR)

sklearn 1.7.2 | output: v4_perubahan


## 1. Muat embedding

In [2]:
acuan = pd.read_parquet(ID_ACUAN)
ids = acuan["Id"].to_numpy(); N = len(ids)
def l2(a):
    a = a.astype(np.float64)
    return a / np.linalg.norm(a, axis=1, keepdims=True)
pre  = l2(np.load(f"{FITUR_DIR}/dino_color_pre.npy"))
post = l2(np.load(f"{FITUR_DIR}/dino_color_post.npy"))
chg  = l2(np.load(f"{FITUR_DIR}/dino_changed_embed.npy"))
diff = l2(post - pre)
PILIHAN = {"changed": chg, "diff": diff, "changed+diff": np.hstack([chg, diff]), "prepost": np.hstack([pre, post])}
X = PILIHAN[FITUR]
print("N =", N, "| fitur:", FITUR, X.shape)

N = 3180 | fitur: changed (3180, 1024)


## 2. PCA → Spectral Clustering

In [3]:
mean_x = X.mean(axis=0)
pca = PCA(PCA_DIM, random_state=SEED)
Z = np.round(pca.fit_transform(X - mean_x), ROUND)

def spectral(Z_in, knn=KNN):
    return SpectralClustering(n_clusters=K, affinity="nearest_neighbors", n_neighbors=knn,
                              assign_labels="cluster_qr", random_state=SEED).fit_predict(Z_in)
event_id = spectral(Z)
ukuran = pd.Series(event_id).value_counts().reindex(range(K), fill_value=0)
centroids = np.vstack([Z[event_id == e].mean(axis=0) for e in range(K)])
jarak_pusat = np.linalg.norm(Z - centroids[event_id], axis=1)
# kedekatan 0-1 dari peringkat jarak di dalam cluster (1 = paling dekat pusat)
kedekatan = np.zeros(N)
for e in range(K):
    ix = np.where(event_id == e)[0]
    rank = np.argsort(np.argsort(jarak_pusat[ix], kind="stable"), kind="stable")
    kedekatan[ix] = 1 - rank / max(len(ix) - 1, 1)
HASH_EVENT = hashlib.sha1(event_id.astype(np.int32).tobytes()).hexdigest()
print("ukuran min/median/maks:", ukuran.min(), int(ukuran.median()), ukuran.max(), "| sha1", HASH_EVENT)

ukuran min/median/maks: 29 71 169 | sha1 e7ea9fd7c665d03c109f917850864d6ecd2881d6


## 3. Gate + perbandingan dengan v3
η² = porsi variasi suatu ukuran yang dijelaskan pembagian cluster. Target: η² perubahan naik, η² scene turun, bias artefak ≤ 0,15.

In [4]:
patch = pd.read_parquet(f"{FITUR_DIR}/patch_change_stats_v3.parquet").set_index("Id").loc[ids]
phys  = pd.read_parquet(f"{FITUR_DIR}/phys_v4.parquet").set_index("Id").loc[ids]
art   = pd.read_parquet(f"{FITUR_DIR}/artifact_and_phys_v3.parquet").set_index("Id").loc[ids]

def eta2(label, x):
    x = np.asarray(x, dtype=float); ok = ~np.isnan(x)
    lab, x = np.asarray(label)[ok], x[ok]
    m = x.mean(); total = ((x - m) ** 2).sum()
    g = pd.Series(x).groupby(lab)
    return float((g.count() * (g.mean() - m) ** 2).sum() / total)

UKURAN_PERUBAHAN = {"frac_chg_p15": patch["frac_chg_p15"], "top10_share": patch["top10_share"], "radial_decay": patch["radial_decay"],
                    "d_dark": phys["d_dark"], "d_water": phys["d_water"], "d_exg_v4": phys["d_exg_v4"],
                    "ash_gain_v4": phys["ash_gain_v4"], "d_entropy": phys["d_entropy"]}
UKURAN_SCENE = {"edge_pre": art["edge_pre"], "exg_pre": art["exg_pre"], "water_pre": art["water_pre"]}
UKURAN_ARTEFAK = {"is_gray (pre+post)": art["is_gray_pre"] + art["is_gray_post"],
                  "scanline (pre+post)": art["scanline_score_pre"] + art["scanline_score_post"],
                  "black_frac (pre+post)": art["black_frac_pre"] + art["black_frac_post"]}

v3_induk = acuan["event_id"].to_numpy().copy()
sub_v3 = pd.read_parquet("v3_revisi2/sub_events.parquet")
v3_induk[pd.Index(ids).get_indexer(sub_v3["Id"])] = sub_v3["event_id_induk"].to_numpy()   # 40 cluster asli v3

tabel = []
for nama, lab in [("v3 (pre+post)", v3_induk), (f"v4 ({FITUR})", event_id)]:
    tabel.append({"versi": nama,
                  "eta2_perubahan": np.mean([eta2(lab, v) for v in UKURAN_PERUBAHAN.values()]),
                  "eta2_scene": np.mean([eta2(lab, v) for v in UKURAN_SCENE.values()]),
                  "bias_artefak_maks": max(eta2(lab, v) for v in UKURAN_ARTEFAK.values())})
tabel = pd.DataFrame(tabel).set_index("versi").round(3)

GATE = {"ari_ulang": adjusted_rand_score(event_id, spectral(Z)),
        "ari_noise": adjusted_rand_score(event_id, spectral(Z + np.random.default_rng(SEED).normal(0, NOISE_SKALA * Z.std(), Z.shape))),
        "ari_knn25": adjusted_rand_score(event_id, spectral(Z, 25)),
        "bias_max": float(tabel.iloc[1]["bias_artefak_maks"]), "min_size": int(ukuran.min()),
        "ari_vs_v3": adjusted_rand_score(v3_induk, event_id)}
GATE["lulus"] = bool(GATE["ari_ulang"] == 1.0 and GATE["ari_noise"] >= GATE_ARI_NOISE
                     and GATE["bias_max"] <= GATE_BIAS_MAX and GATE["min_size"] >= GATE_MIN_SIZE)
print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in GATE.items()})
assert GATE["lulus"], "Gate gagal -> ubah FITUR/K, jangan lanjut"
tabel

{'ari_ulang': 1.0, 'ari_noise': 0.9885, 'ari_knn25': 0.6816, 'bias_max': 0.143, 'min_size': 29, 'ari_vs_v3': 0.1683, 'lulus': True}


,eta2_perubahan,eta2_scene,bias_artefak_maks
versi,,,
v3 (pre+post),0.121,0.224,0.149
v4 (changed),0.163,0.171,0.143


## 4. Simpan hasil clustering (tanpa nama)

In [5]:
joblib.dump(pca, f"{OUTPUT_DIR}/pca.joblib")
np.save(f"{OUTPUT_DIR}/mean_x.npy", mean_x)
np.save(f"{OUTPUT_DIR}/centroids.npy", centroids)
events = pd.DataFrame({"Id": ids, "event_id": event_id.astype(int), "jarak_pusat": np.round(jarak_pusat, 4),
                       "kedekatan": np.round(kedekatan, 4)})
events.to_parquet(f"{OUTPUT_DIR}/events.parquet", index=False)
print(events.head())

             Id  event_id  jarak_pusat  kedekatan
0  000c166d1af6        33       0.4316     0.4961
1  0047c432f3c2        27       0.3028     0.8144
2  006b070b2644        11       0.3523     0.5714
3  0087328f3429        39       0.4241     0.4167
4  00cc6cf40768        18       0.4532     0.0645


## 5. Montase
Tata letak sama dengan v3: 16 pasangan per cluster (kiri = pre, kanan = post), 4 baris atas terdekat pusat, 4 baris bawah acak. Dua versi: asli dan warna dinormalisasi.

In [6]:
GAP_DALAM, GAP_PASANG, GAP_BARIS, TEPI, JUDUL_H = 4, 28, 12, 16, 72
N_BARIS = int(np.ceil((N_DEKAT + N_ACAK) / PASANG_PER_BARIS))
W = 2 * TEPI + PASANG_PER_BARIS * (2 * TILE + GAP_DALAM) + (PASANG_PER_BARIS - 1) * GAP_PASANG
H = JUDUL_H + N_BARIS * TILE + (N_BARIS - 1) * GAP_BARIS + TEPI
try:
    FONT_JUDUL, FONT_SUB, FONT_ID = (ImageFont.truetype("arial.ttf", s) for s in (30, 15, 13))
except OSError:
    try:
        FONT_JUDUL, FONT_SUB, FONT_ID = (ImageFont.truetype("DejaVuSans.ttf", s) for s in (30, 15, 13))
    except OSError:
        FONT_JUDUL = FONT_SUB = FONT_ID = ImageFont.load_default()

def norm_warna(a):
    a = a.astype(np.float32)
    isi = a.max(axis=2) > 10
    if isi.sum() < 100:
        isi = np.ones(a.shape[:2], dtype=bool)
    rata = a[isi].mean(axis=0)
    a = a * (rata.mean() / np.maximum(rata, 1e-6))
    for k in range(3):
        lo, hi = np.percentile(a[..., k][isi], [2, 98])
        a[..., k] = (a[..., k] - lo) / max(hi - lo, 1e-6) * 255
    return np.clip(a, 0, 255).astype(np.uint8)

pilihan = []
for e in range(K):
    ix = np.where(event_id == e)[0]
    urut = ix[np.argsort(jarak_pusat[ix], kind="stable")]
    dekat, sisa = urut[:N_DEKAT], urut[N_DEKAT:]
    acak = np.random.default_rng(SEED + e).choice(sisa, size=min(N_ACAK, len(sisa)), replace=False)
    for pos, (i, jenis) in enumerate([(i, "dekat") for i in dekat] + [(i, "acak") for i in acak]):
        pilihan.append({"event_id": e, "posisi": pos, "Id": ids[i], "jenis": jenis})
pilihan = pd.DataFrame(pilihan)
pilihan.to_csv(f"{OUTPUT_DIR}/montase_ids.csv", index=False)

for e in range(K):
    kanvas = {v: Image.new("RGB", (W, H), "white") for v in ("asli", "norm")}
    for v, kv in kanvas.items():
        d = ImageDraw.Draw(kv)
        d.text((TEPI, 10), f"v4 Cluster {e:02d} - {int(ukuran[e])} foto", fill="black", font=FONT_JUDUL)
        d.text((TEPI, 46), "kiri = pre, kanan = post | 4 baris atas = terdekat pusat, 4 baris bawah = acak"
               + (" | warna dinormalisasi" if v == "norm" else ""), fill=(90, 90, 90), font=FONT_SUB)
    for _, r in pilihan[pilihan["event_id"] == e].iterrows():
        baris, kol = divmod(int(r["posisi"]), PASANG_PER_BARIS)
        x0 = TEPI + kol * (2 * TILE + GAP_DALAM + GAP_PASANG)
        y0 = JUDUL_H + baris * (TILE + GAP_BARIS)
        for j, sisi in enumerate(["pre", "post"]):
            a = np.asarray(Image.open(f"{DATA_DIR}/{r['Id']}/{sisi}.jpg").convert("RGB").resize((TILE, TILE), Image.LANCZOS))
            x = x0 + j * (TILE + GAP_DALAM)
            kanvas["asli"].paste(Image.fromarray(a), (x, y0))
            kanvas["norm"].paste(Image.fromarray(norm_warna(a)), (x, y0))
        for kv in kanvas.values():
            d = ImageDraw.Draw(kv)
            kotak = d.textbbox((x0 + 4, y0 + 4), r["Id"], font=FONT_ID)
            d.rectangle((kotak[0] - 2, kotak[1] - 2, kotak[2] + 2, kotak[3] + 2), fill="white")
            d.text((x0 + 4, y0 + 4), r["Id"], fill="black", font=FONT_ID)
    kanvas["asli"].save(f"{OUTPUT_DIR}/montase/cluster_{e:02d}.jpg", quality=JPEG_Q)
    kanvas["norm"].save(f"{OUTPUT_DIR}/montase/cluster_{e:02d}_norm.jpg", quality=JPEG_Q)
print("montase:", len(os.listdir(f"{OUTPUT_DIR}/montase")), "file")

montase: 80 file


## 6. Lembar penamaan kosong + meta.json
Format sama dengan `v3/lembar_bersama.csv`. `kelas_final`, `yakin`, `alasan` diisi tim dari montase (FAQ no. 7), bukan dari leaderboard.

In [7]:
lembar = pd.DataFrame({"event_id": range(K), "n": ukuran.to_numpy(), "montase": [f"montase/cluster_{e:02d}.jpg" for e in range(K)],
                       "kelas_final": "", "yakin": "", "alasan": ""})
lembar.to_csv(f"{OUTPUT_DIR}/lembar_bersama.csv", index=False, encoding="utf-8-sig", lineterminator="\r\n")

meta = {"versi": "v4_perubahan", "oleh": "Fahmi", "tanggal": datetime.now().strftime("%Y-%m-%d %H:%M"),
        "input": {"fitur": FITUR_DIR, "embedding": f"{FITUR} (dino_changed_embed = token post dibobot peta perubahan)"},
        "pengaturan": {"metode": "spectral", "affinity": "nearest_neighbors", "assign_labels": "cluster_qr",
                       "fitur": FITUR, "K": K, "knn": KNN, "pca": PCA_DIM, "round": ROUND, "seed": SEED},
        "gate": {k: (round(v, 4) if isinstance(v, float) else v) for k, v in GATE.items()},
        "perbandingan_v3": tabel.reset_index().to_dict("records"),
        "ukuran_cluster": {str(e): int(n) for e, n in ukuran.items()},
        "event_id_sha1": HASH_EVENT,
        "versi_lib": {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__, "scikit-learn": sklearn.__version__},
        "catatan": "Cluster dari fitur perubahan. Nama kelas belum diisi: diisi tim dari montase (FAQ no. 7)."}
with open(f"{OUTPUT_DIR}/meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)
print("selesai:", sorted(os.listdir(OUTPUT_DIR)))

selesai: ['centroids.npy', 'events.parquet', 'lembar_bersama.csv', 'mean_x.npy', 'meta.json', 'montase', 'montase_ids.csv', 'pca.joblib']
